In [ ]:
#@title Install
from google.colab import drive
drive.mount('/content/drive')
!pip install -q librosa soundfile scipy matplotlib tqdm requests
!pip install -q --force-reinstall --no-deps "numpy==1.26.4"


In [ ]:
#@title Configuration
import os, json, random
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/bearing_protocol')
DOWNLOAD_DIR = ROOT/'paderborn_raw'
DATA_DIR = ROOT/'data_pb_5khz'
for p in (DOWNLOAD_DIR, DATA_DIR/'train', DATA_DIR/'test'): p.mkdir(parents=True, exist_ok=True)

SPEC_ID = 'pb_5khz_inferno_pg03_v4'

SEGMENT_DURATION = 2.0
OVERLAP_RATIO    = 0.5
MAX_SEGMENTS_PER_FILE = 20
TARGET_SR        = 16000
STFT_NPERSEG     = 2048
STFT_MAX_FREQ    = 5000.0
STFT_IMG_W, STFT_IMG_H = 640, 480
STFT_CMAP        = 'inferno'
STFT_GAMMA       = 0.3
BANDPASS_LOW, BANDPASS_HIGH = 1000.0, 10000.0
FREQ_BAND_WIDTH  = 5.0

N_BALLS, D_BALL, D_PITCH = 8, 6.75, 28.5
_r = D_BALL/D_PITCH
BPFO_COEFF = (N_BALLS/2)*(1-_r)
BPFI_COEFF = (N_BALLS/2)*(1+_r)

TRAIN_BEARINGS = ['KA01','KA03','KA05','KA06','KA07','KA08','KA09',
                  'KI01','KI03','KI05','KI07','KI08',
                  'K001','K002','K003','K004','K005']
TEST_BEARINGS  = ['KI04','KI14','KI16','KI17','KI18','KI21',
                  'KA04','KA15','KA16','KA22','KA30','K006']
ALL_BEARINGS = TRAIN_BEARINGS + TEST_BEARINGS
CLASSES = ['Healthy','Inner_Ring_Fault','Outer_Ring_Fault']

SEED = 42
random.seed(SEED); np.random.seed(SEED)

print(f'bin width {64000/STFT_NPERSEG:.2f} Hz, {int(STFT_MAX_FREQ/(64000/STFT_NPERSEG))} bins to 5 kHz')
print(f'BPFO {BPFO_COEFF:.4f} f_r   BPFI {BPFI_COEFF:.4f} f_r')
print(f'train bearings {len(TRAIN_BEARINGS)}   test bearings {len(TEST_BEARINGS)}')

In [ ]:
#@title Signal processing
import requests, scipy.io, librosa, soundfile as sf
from tqdm import tqdm
from scipy import stats as scipy_stats
from scipy.signal import butter, filtfilt, hilbert, stft as scipy_stft
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

HEADERS  = {'User-Agent': 'Mozilla/5.0'}
BASE_URL = "https://groups.uni-paderborn.de/kat/BearingDataCenter"

def get_label(n):
    if n.startswith('K0'): return 'Healthy'
    if n.startswith('KI'): return 'Inner_Ring_Fault'
    if n.startswith('KA'): return 'Outer_Ring_Fault'

def download_file(url, filepath):
    try:
        r = requests.get(url, stream=True, headers=HEADERS, timeout=60)
        if r.status_code != 200: return False
        total = int(r.headers.get('content-length', 0))
        with open(filepath, 'wb') as f, tqdm(desc=os.path.basename(filepath),
                                             total=total, unit='iB', unit_scale=True) as bar:
            for d in r.iter_content(1024 * 1024):
                bar.update(len(d)); f.write(d)
        return True
    except Exception as e:
        print(f"  Error: {e}"); return False

def load_mat(p):
    mat=scipy.io.loadmat(str(p), squeeze_me=False)
    key=[k for k in mat if not k.startswith('__')][0]; data=mat[key]; r={}
    xa=data['X'][0,0]
    for j in range(xa.shape[1]):
        x=xa[0,j]; raster=str(x['Raster'].flat[0]) if x['Raster'].size>0 else ''
        d=x['Data'].flatten()
        if len(d)>1:
            dt=d[1]-d[0]; sr=1.0/dt if dt>0 else 0
            if 'HostService' in raster: r['vib_sr']=sr
            elif 'Mech_4kHz' in raster:  r['mech_sr']=sr
    ya=data['Y'][0,0]
    for j in range(ya.shape[1]):
        y=ya[0,j]; nm=str(y['Name'].flat[0]) if y['Name'].size>0 else ''
        d=y['Data'].flatten()
        if nm=='vibration_1': r['vibration']=d
        elif nm=='speed':     r['speed_rpm']=d
    return r

def vibration_to_audio(signal, source_sr=64000, target_sr=16000):
    signal = signal - np.mean(signal)
    rs = librosa.resample(signal, orig_sr=source_sr, target_sr=target_sr)
    m = np.max(np.abs(rs))
    if m > 0: rs = rs/m*0.95
    return rs.astype(np.float32)

def gen_stft_image(vib_segment, vib_sr, output_path, max_freq=None):
    max_freq = STFT_MAX_FREQ if max_freq is None else max_freq
    f, t, Zxx = scipy_stft(vib_segment, fs=vib_sr, nperseg=STFT_NPERSEG,
                           noverlap=STFT_NPERSEG // 2)
    freq_mask = f <= max_freq
    f = f[freq_mask]; Zxx = Zxx[freq_mask, :]
    fig, ax = plt.subplots(1, 1, figsize=(STFT_IMG_W/100, STFT_IMG_H/100), dpi=100)
    ax.pcolormesh(t, f, np.abs(Zxx), shading='gouraud', cmap=STFT_CMAP,
                  norm=PowerNorm(gamma=STFT_GAMMA))
    ax.set_ylim(0, max_freq); ax.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

def bandpass_filter(s,lo,hi,sr,order=5):
    nyq=0.5*sr
    b,a=butter(order,[lo/nyq,min(hi/nyq,0.99)],btype='band'); return filtfilt(b,a,s)

def envelope_spectrum(s,sr,lo,hi):
    env_t=np.abs(hilbert(bandpass_filter(s,lo,hi,sr)))
    e=env_t-np.mean(env_t)
    return np.fft.rfftfreq(len(e),d=1.0/sr), np.abs(np.fft.rfft(e)), env_t

def extract_features_adaptive(vib, vib_sr, speed_rpm):
    rms=float(np.sqrt(np.mean(vib**2)))
    kurtosis=float(scipy_stats.kurtosis(vib, fisher=True))
    peak=float(np.max(np.abs(vib)))
    crest=float(peak/rms) if rms>0 else 0.0
    f_shaft=speed_rpm/60.0
    bpfo_hz=BPFO_COEFF*f_shaft; bpfi_hz=BPFI_COEFF*f_shaft
    try:
        ef, em, env_t = envelope_spectrum(vib, vib_sr, BANDPASS_LOW, BANDPASS_HIGH)
        def band(c,bw=FREQ_BAND_WIDTH):
            m=(ef>=c-bw)&(ef<=c+bw)
            return float(np.sum(em[m]**2)) if np.any(m) else 0.0
        bpfo_1x, bpfo_2x, bpfo_3x = band(bpfo_hz), band(2*bpfo_hz), band(3*bpfo_hz)
        bpfi_1x, bpfi_2x          = band(bpfi_hz), band(2*bpfi_hz)
        fault=(ef>=20)&(ef<=500)
        total=max(float(np.sum(em[fault]**2)),1e-10)
        bpfo_ratio=(bpfo_1x+bpfo_2x)/total
        bpfi_ratio=(bpfi_1x+bpfi_2x)/total
        bpfi_sideband_ratio=band(bpfi_hz-f_shaft)/total
        bpfo_harmonic_ratio=(bpfo_2x+bpfo_3x)/total
        ebf=em[fault]
        peak_prominence=float(np.max(ebf)/(np.median(ebf)+1e-12)) if len(ebf) else 0.0
        modulation_index=float(np.std(env_t)/(np.mean(env_t)+1e-12))
        fault_band_fraction=float(np.sum(ebf**2)/max(float(np.sum(em**2)),1e-10))
        char=(bpfo_1x+bpfo_2x+bpfo_3x+bpfi_1x+bpfi_2x+band(bpfi_hz-f_shaft))
        char_energy_frac=float(char/total)
        mn=max(1,int(20*len(ef)/(vib_sr/2))); mx=int(500*len(ef)/(vib_sr/2))
        rng=em[mn:mx]
        dom=float(ef[np.argmax(rng)+mn]) if len(rng)>0 else 0.0
        epos=em[fault][em[fault]>0]
        flat=float(np.exp(np.mean(np.log(epos+1e-10)))/(np.mean(epos)+1e-10)) if len(epos)>10 else 1.0
    except Exception:
        bpfo_ratio=bpfi_ratio=0.0; dom=0.0; flat=1.0
        bpfi_sideband_ratio=bpfo_harmonic_ratio=0.0
        peak_prominence=modulation_index=fault_band_fraction=char_energy_frac=0.0
    return {'rms':round(rms,4),'kurtosis':round(kurtosis,2),
            'crest_factor':round(crest,2),'speed_rpm':round(speed_rpm,1),
            'f_shaft_hz':round(f_shaft,2),'bpfo_hz':round(bpfo_hz,1),
            'bpfi_hz':round(bpfi_hz,1),'env_dominant_freq_hz':round(dom,1),
            'env_spectral_flatness':round(flat,4),
            'env_bpfo_ratio':round(bpfo_ratio,4),'env_bpfi_ratio':round(bpfi_ratio,4),
            'env_bpfi_sideband_ratio':round(bpfi_sideband_ratio,4),
            'env_bpfo_harmonic_ratio':round(bpfo_harmonic_ratio,4),
            'env_peak_prominence':round(peak_prominence,3),
            'env_modulation_index':round(modulation_index,4),
            'env_fault_band_fraction':round(fault_band_fraction,4),
            'env_char_energy_frac':round(char_energy_frac,4)}

In [ ]:
#@title Download the Paderborn recordings
print(f'mat files on disk: {len(list(DOWNLOAD_DIR.rglob("*.mat")))}')

for b in ALL_BEARINGS:
    rar_path = DOWNLOAD_DIR / f"{b}.rar"
    bearing_dir = DOWNLOAD_DIR / b
    if bearing_dir.exists() and list(bearing_dir.glob("*.mat")):
        continue
    if not rar_path.exists():
        print(f"downloading {b}")
        if not download_file(f"{BASE_URL}/{b}.rar", rar_path):
            print(f"  {b} download failed"); continue

!apt-get install -qq unrar 2>&1 | tail -1
for b in ALL_BEARINGS:
    rar_path = DOWNLOAD_DIR / f"{b}.rar"
    bearing_dir = DOWNLOAD_DIR / b
    if rar_path.exists() and not bearing_dir.exists():
        !cd {DOWNLOAD_DIR} && unrar x -y {b}.rar > /dev/null 2>&1

files_by_bearing = defaultdict(list)
for b in ALL_BEARINGS:
    bdir = DOWNLOAD_DIR / b
    if bdir.exists():
        files_by_bearing[b] = sorted(bdir.glob("*.mat"))

n_files = sum(len(v) for v in files_by_bearing.values())
print(f'\n{n_files} mat files across {len(files_by_bearing)} bearings')
missing = [b for b in ALL_BEARINGS if not files_by_bearing.get(b)]
assert not missing, f'missing bearings: {missing}'

In [ ]:
#@title Segments, audio, spectrograms and measured quantities
all_train, all_test = [], []

def process(bearing, split_dir, out):
    label=get_label(bearing)
    for mf in files_by_bearing[bearing]:
        try: ch=load_mat(mf)
        except Exception: continue
        vib=ch.get('vibration')
        if vib is None: continue
        vib_sr=ch.get('vib_sr',64000); spd=ch.get('speed_rpm'); mech=ch.get('mech_sr',4000)
        n=int(SEGMENT_DURATION*vib_sr); hop=int(n*(1-OVERLAP_RATIO))
        for i in range(min(MAX_SEGMENTS_PER_FILE,(len(vib)-n)//hop+1)):
            st=i*hop; en=st+n; seg=vib[st:en]
            if len(seg)<n: continue
            ms=int(st/vib_sr*mech); me=int(en/vib_sr*mech)
            sp=(float(np.mean(spd[ms:min(me,len(spd))]))
                if spd is not None and len(spd)>ms else 900.0)
            an=f'{bearing}_{mf.stem}_{i:03d}.wav'; sn=f'{bearing}_{mf.stem}_{i:03d}_stft.png'
            if not (split_dir/an).exists():
                sf.write(str(split_dir/an), vibration_to_audio(seg, vib_sr, TARGET_SR), TARGET_SR)
            if not (split_dir/sn).exists():
                gen_stft_image(seg, vib_sr, split_dir/sn)
            out.append({'audio':an,'image':str(split_dir/sn),'label':label,
                        'bearing':bearing,
                        'features':extract_features_adaptive(seg,vib_sr,sp)})

for b in tqdm(TRAIN_BEARINGS, desc='train'): process(b, DATA_DIR/'train', all_train)
for b in tqdm(TEST_BEARINGS,  desc='test'):  process(b, DATA_DIR/'test',  all_test)

print(f"\ntrain {len(all_train)}  {dict(Counter(s['label'] for s in all_train))}")
print(f"test  {len(all_test)}  {dict(Counter(s['label'] for s in all_test))}")
n_wav = sum(len(list((DATA_DIR/sp).glob('*.wav'))) for sp in ('train','test'))
n_png = sum(len(list((DATA_DIR/sp).glob('*_stft.png'))) for sp in ('train','test'))
assert n_wav == n_png == len(all_train)+len(all_test), 'file count mismatch'

manifest={'train':all_train,'test':all_test}
_mp=DATA_DIR/'data_manifest_v4.json'
if _mp.exists():
    _old=json.load(open(_mp))
    _sig=lambda m: {sp: sorted((r['audio'], r['label'], json.dumps(r['features'], sort_keys=True)) for r in m[sp])
                    for sp in ('train','test')}
    assert _sig(_old)==_sig(manifest), 'data_manifest_v4.json differs from the data built here'
    print('manifest matches the one on disk')
else:
    json.dump(manifest, open(_mp,'w')); print('manifest written:', _mp.name)

In [ ]:
#@title Thresholds and separability
def sep(feat,target,rows):
    pos=np.array([r['features'][feat] for r in rows if r['label']==target])
    neg=np.array([r['features'][feat] for r in rows if r['label']!=target])
    if len(pos)==0 or len(neg)==0: return float('nan')
    x=np.concatenate([pos,neg]); o=x.argsort(kind='mergesort')
    ranks=np.empty(len(x)); ranks[o]=np.arange(1,len(x)+1)
    _,inv,cnt=np.unique(x,return_inverse=True,return_counts=True)
    s=np.zeros(len(cnt)); np.add.at(s,inv,ranks); ranks=(s/cnt)[inv]
    n1,n2=len(pos),len(neg)
    a=(ranks[:n1].sum()-n1*(n1+1)/2)/(n1*n2)
    return max(a,1-a)

TARGETS={'env_bpfi_ratio':'Inner_Ring_Fault','env_bpfo_ratio':'Outer_Ring_Fault',
         'env_bpfo_harmonic_ratio':'Outer_Ring_Fault',
         'env_char_energy_frac':'Healthy',
         'env_bpfi_sideband_ratio':'Inner_Ring_Fault'}

thresholds={}
for q in TARGETS:
    v=np.array([r['features'][q] for r in all_train])
    thresholds[q]=[round(float(np.percentile(v,33.333)),4),
                   round(float(np.percentile(v,66.667)),4)]
separability={q:sep(q,t,all_train) for q,t in TARGETS.items()}

print(f"{'quantity':28}{'lower':>9}{'upper':>9}{'separability':>14}")
for q in TARGETS:
    lo,hi=thresholds[q]
    print(f'{q:28}{lo:9.4f}{hi:9.4f}{separability[q]:14.3f}')

spec={'spec_id':SPEC_ID,'dataset':'PB',
      'segment_duration_s':SEGMENT_DURATION,'overlap_ratio':OVERLAP_RATIO,
      'max_segments_per_file':MAX_SEGMENTS_PER_FILE,
      'vib_sr_hz':64000,'audio_target_sr_hz':TARGET_SR,
      'stft_nperseg':STFT_NPERSEG,'stft_noverlap':STFT_NPERSEG//2,
      'freq_resolution_hz_per_bin':64000/STFT_NPERSEG,'stft_max_freq_hz':STFT_MAX_FREQ,
      'cmap':STFT_CMAP,'norm':f'PowerNorm(gamma={STFT_GAMMA})',
      'scale':'linear_magnitude','shading':'gouraud','img_w':STFT_IMG_W,'img_h':STFT_IMG_H,
      'n_balls':N_BALLS,'d_ball_mm':D_BALL,'d_pitch_mm':D_PITCH,
      'bpfo_coeff':BPFO_COEFF,'bpfi_coeff':BPFI_COEFF,
      'bandpass_hz':[BANDPASS_LOW,BANDPASS_HIGH],'freq_band_width_hz':FREQ_BAND_WIDTH,
      'train_bearings':TRAIN_BEARINGS,'test_bearings':TEST_BEARINGS,'classes':CLASSES,
      'n_train':len(all_train),'n_test':len(all_test),
      'features':sorted(all_train[0]['features'].keys()),
      'thresholds':thresholds,'separability':separability}
_sp=DATA_DIR/'spec_v4.json'
if _sp.exists():
    _old=json.load(open(_sp))['thresholds']
    for q,(lo,hi) in thresholds.items():
        assert _old[q]==[lo,hi], f'{q}: thresholds {lo}/{hi} differ from spec_v4.json {_old[q]}'
    print('\nthresholds match spec_v4.json')
else:
    json.dump(spec, open(_sp,'w'), indent=2); print('\nspec ->', _sp)